# Lab 1-04：直接打 Chat API

**目標**：不靠任何框架，用 `requests` 送出一個 HTTP 請求跟模型對話，看清楚來回的 JSON 長什麼樣。

**你會學到**
- Chat API 的請求只是一個 `messages` 陣列
- system / user / assistant 三種角色各管什麼
- 為什麼模型記不住上一句：記憶是你每次重送的
- 串流（streaming）怎麼一個字一個字收

**前置**：本機裝好 [Ollama](https://ollama.com/download)，並在終端機執行：

```bash
ollama pull qwen2.5:3b
```

想改用雲端，只要在 `.env` 改 `LOCAL_BASE_URL`、`LOCAL_MODEL`（下一本會做自動切換）。

**對應投影片**：「messages JSON 實物」「為什麼記不住對話」「溫度只管隨機性」

## Step 1：確認 Ollama 有在跑

Ollama 提供跟 OpenAI 一樣格式的端點（`/v1/...`），所以後面所有程式碼換成雲端服務也能用。

In [1]:
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv()
BASE_URL = os.getenv("LOCAL_BASE_URL", "http://localhost:11434/v1")
MODEL = os.getenv("LOCAL_MODEL", "qwen2.5:3b")

models = requests.get(f"{BASE_URL}/models", timeout=5).json()
print("本機有的模型：", [m["id"] for m in models["data"]])
print("這本 notebook 用：", MODEL)

本機有的模型： ['qwen2.5:3b']
這本 notebook 用： qwen2.5:3b


## Step 2：最原始的一次呼叫

請求就是一個 JSON：要用哪個模型、對話內容 `messages`、還有幾個抽樣參數。

In [2]:
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "用一句話介紹魯夫。"}],
    "temperature": 0,
}
resp = requests.post(f"{BASE_URL}/chat/completions", json=payload, timeout=120)
data = resp.json()
print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "id": "chatcmpl-236",
  "object": "chat.completion",
  "created": 1794997683,
  "model": "qwen2.5:3b",
  "system_fingerprint": "fp_ollama",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "魯夫是《海賊王》中的主角之一，以其智慧和正义感著称。"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 36,
    "prompt_tokens_details": {
      "cached_tokens": 35
    },
    "completion_tokens": 22,
    "total_tokens": 58
  }
}


回應裡要看三個地方：
- `choices[0].message.content`：模型的回答
- `choices[0].finish_reason`：`stop` 是自己講完，`length` 是被字數上限切斷
- `usage`：這次用了多少 token（雲端服務就是照這個收費）

In [3]:
print("回答：", data["choices"][0]["message"]["content"])
print("結束原因：", data["choices"][0]["finish_reason"])
print("token 用量：", data["usage"])

回答： 魯夫是《海賊王》中的主角之一，以其智慧和正义感著称。
結束原因： stop
token 用量： {'prompt_tokens': 36, 'prompt_tokens_details': {'cached_tokens': 35}, 'completion_tokens': 22, 'total_tokens': 58}


小模型對海賊王的了解很有限，回答很可能是錯的。這不是程式的問題，是上一本講的幻覺；模組 3 的 RAG 會處理它。

把它包成一個小函式，後面方便用。

In [4]:
def chat(messages: list[dict], temperature: float = 0, **extra) -> dict:
    payload = {"model": MODEL, "messages": messages, "temperature": temperature, **extra}
    resp = requests.post(f"{BASE_URL}/chat/completions", json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()


def say(messages: list[dict], **kw) -> str:
    return chat(messages, **kw)["choices"][0]["message"]["content"]

## Step 3：system message 設定角色與規則

`system` 放的是「整段對話都要遵守」的指示；`user` 是這一輪的問題。

In [5]:
question = {"role": "user", "content": "今天天氣如何？"}

print("--- 沒有 system")
print(say([question]))

print("--- 有 system")
print(say([
    {"role": "system", "content": "你是草帽海賊團的航海士娜美。回答一律不超過兩句，語氣直接。"},
    question,
]))

--- 沒有 system
我很抱歉，由于我是一个云端存在的模型，我没有直接获取实时天气信息的功能。我无法提供你所在地区的具体天气情况。你可以查阅当地的天气预报或者使用相关的天气应用查看。如果你告诉我你所在的城市，我可以提供一些建议来帮助你查找天气信息。
--- 有 system
今天風勢不小，海浪也不小。


## Step 4：溫度——同一題問三次

In [6]:
msg = [{"role": "user", "content": "幫草帽海賊團的新船取一個名字，只回名字。"}]
for T in [0, 1.2]:
    print(f"T = {T}：", [say(msg, temperature=T) for _ in range(3)])

T = 0： ['海賊之風', '海賊之風', '海賊之風']
T = 1.2： ['飛鼠艇', '自由號', '白星船队']


T = 0 三次幾乎一樣；T 調高，每次都不同。要輸出 JSON 或 SQL 時，溫度一律設 0。

## Step 5：模型記不住你

先告訴它名字，下一個請求再問。

In [7]:
print(say([{"role": "user", "content": "我叫索隆，請記住。"}]))
print("---")
print(say([{"role": "user", "content": "我叫什麼名字？"}]))

好的，我會記住你叫索隆。有什么我可以帮助你解答或讨论的吗？无论是关于你的问题，还是其他你感兴趣的话题，我都很乐意交谈。
---
您目前还没有告诉我您的名字，您可以告诉我，您可以称呼我为Qwen。


第二次請求裡根本沒有名字，模型當然不知道。模型是一個函式：輸入 messages、輸出下一段字，**兩次呼叫之間什麼都不留**。

「記憶」其實是你把整段歷史一起重送：

In [8]:
history = [{"role": "user", "content": "我叫索隆，請記住。"}]
history.append({"role": "assistant", "content": say(history)})
history.append({"role": "user", "content": "我叫什麼名字？"})
print(say(history))
print("這次送出的 messages：")
print(json.dumps(history, ensure_ascii=False, indent=2))

你叫索隆。
這次送出的 messages：
[
  {
    "role": "user",
    "content": "我叫索隆，請記住。"
  },
  {
    "role": "assistant",
    "content": "好的，我會記住你叫索隆。有什么我可以帮助你解答或讨论的吗？无论是关于你的问题，还是其他你感兴趣的话题，我都很乐意交谈。"
  },
  {
    "role": "user",
    "content": "我叫什麼名字？"
  }
]


實務後果：對話越長，每次重送的 token 越多，成本和延遲一起上升。
所以真實系統會做摘要或只留最近幾輪；模組 5 的 agent 也要處理這件事。

## Step 6：被字數上限切斷

In [9]:
r = chat([{"role": "user", "content": "詳細介紹草帽海賊團每一位成員。"}], max_tokens=30)
print(r["choices"][0]["message"]["content"])
print("結束原因：", r["choices"][0]["finish_reason"])

草帽海賊團，是《海賊王》這部動畫及小說中的重要角色團體，團長是蒙
結束原因： length


`finish_reason == "length"` 代表回答沒講完。程式裡要檢查它，不然會把半截 JSON 當成完整結果。

## Step 7：串流——邊生成邊顯示

長回答要等很久才一次回來。加上 `"stream": true`，伺服器會一小段一小段送回來，每段是一行 `data: {...}`。

In [10]:
def stream_chat(messages: list[dict], temperature: float = 0):
    payload = {"model": MODEL, "messages": messages, "temperature": temperature, "stream": True}
    with requests.post(f"{BASE_URL}/chat/completions", json=payload, stream=True, timeout=120) as resp:
        for raw in resp.iter_lines():
            line = raw.decode("utf-8")  # 先拿 bytes 再自己解碼，中文才不會被切壞
            if not line.startswith("data: "):
                continue
            body = line.removeprefix("data: ")
            if body == "[DONE]":
                break
            delta = json.loads(body)["choices"][0]["delta"]
            if delta.get("content"):
                yield delta["content"]


for piece in stream_chat([{"role": "user", "content": "用三句話介紹喬巴。"}]):
    print(piece, end="", flush=True)

喬巴，又名喬巴山，是中國雲南省的一座著名山峰，以其壯觀的自然景觀和豐富的生態資源而聞名。

`stream_chat` 是一個生成器（generator）：每收到一小段就 `yield` 出去，呼叫端拿到就印，不用等全部結束。

## Step 8：同樣的事，用官方 SDK 寫

理解了 HTTP 長什麼樣之後，平常就用 SDK，少寫很多樣板。`base_url` 換掉就能連任何 OpenAI 相容的服務。

In [11]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key="ollama")
r = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "用一句話介紹索隆。"}], temperature=0
)
print(r.choices[0].message.content)

索隆是《海贼王》中的主角，一位坚定的剑士，誓言消灭所有妨碍路飞的敌人。


## 試試看

1. 在 Step 3 的 system message 加一條「只能用英文回答」，再用中文問問題，看它聽誰的。
2. 寫一個迴圈版的聊天：用 `input()` 收問題，每輪把回答 append 進 `history`，印出每輪的 `usage`，觀察 token 怎麼增加。
3. 如果你有推理模型（例如 `ollama pull qwen3:4b`），把 `MODEL` 換掉，看看回答前面多出來的思考段落。

## 小結

- Chat API＝送一個 `messages` 陣列，拿回一段文字和 token 用量。
- 模型不記得任何事；記憶是你每次重送歷史。
- 串流就是逐行讀 `data:`。下一本：把這些包成一個「本地掛了會自動切雲端」的客戶端。